# Lenguajes y Autómatas II (SCD-1016)
## Sesión 18 · Arreglos y Verificación de Límites Estáticos

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo calcula el compilador cuánta memoria reservar para un arreglo y hasta dónde puede leer?
>
> **Objetivo de la sesión:** Calcular el tamaño total de un arreglo a partir de su tipo y su número de elementos, y validar índices contra esos límites.

**Entregable:** tus funciones bytes_arreglo() e indice_en_rango() en artesano_sesion18_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion18.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes el nombre de un tipo y el número de elementos; bytes_arreglo() devuelve el total de bytes e indice_en_rango() devuelve 1 si el índice puede leerse o 0 si no. El harness prueba tipos de distinto ancho y los bordes exactos del rango.

Al final de la sesión tendrás: **Dos funciones que dimensionan arreglos y rechazan índices fuera de rango, con pruebas en verde.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 18](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion18.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion18.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion18.h"

long bytes_arreglo(const char *tipo, int elementos);
int indice_en_rango(int indice, int elementos);
```

- Tipos válidos: char, short, int, float, double y puntero; otro nombre devuelve -1.
- Un número negativo de elementos es inválido y devuelve -1; cero elementos pesa cero bytes.
- El índice válido más alto es elementos - 1, nunca elementos.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion18_c.c` junto con `artesano_sesion18_c.c`
> (o `ia_sesion18_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `int[10]` | 40 |
| 2 | `double[3]` | 24 |
| 3 | `int[-4]` | -1 |
| 4 | `indice 0 de 5` | 1 |
| 5 | `indice 5 de 5` | 0 |
| 6 | `indice negativo` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 18 - Tamano de arreglo y limites ===
Caso                               esperado:              obtenido:              resultado
[ 1] int[10]                            esperado: 40           obtenido: 40           OK
[ 2] double[3]                          esperado: 24           obtenido: 24           OK
[ 3] int[-4]                            esperado: -1           obtenido: -1           OK
[ 4] indice 0 de 5                      esperado: 1            obtenido: 1            OK
[ 5] indice 5 de 5                      esperado: 0            obtenido: 0            OK
[ 6] indice negativo                    esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion18.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] bytes_arreglo() multiplica elementos por el ancho correcto y rechaza dimensiones negativas.
- [ ] indice_en_rango() acepta el primer elemento y rechaza el índice igual al número de elementos.
- [ ] Los índices negativos se rechazan sin desbordar la comparación.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion18_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion18.h
/* api_sesion18.h - contrato de la Sesion 18
 * Tamaño de arreglo y límites
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION18_H
#define API_SESION18_H

long bytes_arreglo(const char *tipo, int elementos);
int indice_en_rango(int indice, int elementos);

#endif /* API_SESION18_H */


Writing api_sesion18.h


In [3]:
%%writefile artesano_sesion18_c.c
/* artesano_sesion18_c.c - Fase 1: El Artesano (sin IA)
 * Tamaño de arreglo y límites.
 *
 * Implementa lo que declara api_sesion18.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion18.h"
#include <stdio.h>
#include <string.h>

// Tabla con el ancho en bytes de cada tipo que acepta el contrato
typedef struct {
    const char *nombre;
    long ancho;
} AnchoTipo;

static const AnchoTipo anchos[] = {
    {"char", 1}, {"short", 2}, {"int", 4},
    {"float", 4}, {"double", 8}, {"puntero", 8}
};

long bytes_arreglo(const char *tipo, int elementos) {
    // Sin tipo o con elementos negativos el arreglo no es válido
    if (tipo == NULL || elementos < 0) {
        return -1;
    }

    // Buscamos el tipo en la tabla y multiplicamos por su ancho
    for (size_t i = 0; i < sizeof(anchos) / sizeof(anchos[0]); i++) {
        if (strcmp(anchos[i].nombre, tipo) == 0) {
            return anchos[i].ancho * (long)elementos;
        }
    }

    // Si no lo encontramos, el tipo es desconocido
    return -1;
}

int indice_en_rango(int indice, int elementos) {
    // El primer índice es 0 y el último es elementos - 1
    if (indice < 0) {
        return 0;
    }
    if (indice >= elementos) {
        return 0;
    }
    return 1;
}

Writing artesano_sesion18_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion18_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [4]:
%%writefile ia_sesion18_c.c
/* ia_sesion18_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion18.h. No puede traer main().
 */
#include "api_sesion18.h"
#include <string.h>

static long ancho_de(const char *tipo) {
    if (strcmp(tipo, "char") == 0) return 1;
    if (strcmp(tipo, "short") == 0) return 2;
    if (strcmp(tipo, "int") == 0 || strcmp(tipo, "float") == 0) return 4;
    if (strcmp(tipo, "double") == 0 || strcmp(tipo, "puntero") == 0) return 8;
    return 0;
}

long bytes_arreglo(const char *tipo, int elementos) {
    if (tipo == NULL || elementos < 0) {
        return -1;
    }
    long ancho = ancho_de(tipo);
    if (ancho == 0) {
        return -1;
    }
    return ancho * elementos;
}

int indice_en_rango(int indice, int elementos) {
    return (unsigned)indice < (unsigned)elementos;
}

Writing ia_sesion18_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion18_c.c
/* harness_sesion18_c.c - auditoria automatica de la Sesion 18
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion18_c.c artesano_sesion18_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion18.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 18 - Tamano de arreglo y limites ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { long r = bytes_arreglo("int", 10); verificar_int("int[10]", (long)(40), (long)(r)); }
    { long r = bytes_arreglo("double", 3); verificar_int("double[3]", (long)(24), (long)(r)); }
    { long r = bytes_arreglo("int", -4); verificar_int("int[-4]", (long)(-1), (long)(r)); }
    { int r = indice_en_rango(0, 5); verificar_int("indice 0 de 5", (long)(1), (long)(r)); }
    { int r = indice_en_rango(5, 5); verificar_int("indice 5 de 5", (long)(0), (long)(r)); }
    { int r = indice_en_rango(-1, 5); verificar_int("indice negativo", (long)(0), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion18.md" "\n");
    return 0;
}


Writing harness_sesion18_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion18_c.c artesano_sesion18_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 18 - Tamano de arreglo y limites ===
Caso                               esperado:              obtenido:              resultado
[ 1] int[10]                            esperado: 40           obtenido: 40           OK
[ 2] double[3]                          esperado: 24           obtenido: 24           OK
[ 3] int[-4]                            esperado: -1           obtenido: -1           OK
[ 4] indice 0 de 5                      esperado: 1            obtenido: 1            OK
[ 5] indice 5 de 5                      esperado: 0            obtenido: 0            OK
[ 6] indice negativo                    esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion18.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion18_c.c ia_sesion18_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 18 - Tamano de arreglo y limites ===
Caso                               esperado:              obtenido:              resultado
[ 1] int[10]                            esperado: 40           obtenido: 40           OK
[ 2] double[3]                          esperado: 24           obtenido: 24           OK
[ 3] int[-4]                            esperado: -1           obtenido: -1           OK
[ 4] indice 0 de 5                      esperado: 1            obtenido: 1            OK
[ 5] indice 5 de 5                      esperado: 0            obtenido: 0            OK
[ 6] indice negativo                    esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion18.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion18.md` a `auditorias/auditoria_sesion18.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
